기존에 했던 방법은 pca축을 하나씩 해놓고 각 pc축들을 독립적으로 테스트해봤음. 이 내용을 생각해보면 왜 pc축을 해놨는지 의문이 드는 실험임.   
내가 왜 이런 생각을 하지 못했던건지 모르겠지만, 이런식으로 독립적으로 실험을 하면 왜 pc축들을 직교하게 하는 축을 찾았을까 라는 생각이 도출되어야 한다고 생각함.   
왜 이런 단계를 생각하지 못했는지 생각해봐야 할 필요가 있음.


# Beer 실험 v3: 누적 PCR (PC1 → PC1+PC2 → ... )로 관능(50개) 예측 곡선 만들기
이 노트북은 **“PCA가 분산 큰 순서대로 축을 만든다”**는 PCA의 설계와 일관되게,
PC1부터 하나씩 **누적**해서(1개, 2개, 3개, …) 회귀를 학습시키는 **PCR learning curve** 실험을 구현합니다.

## 왜 이 실험이 필요한가?
- 이전 실험(PC를 하나씩 독립적으로 1D 회귀)은 “**각 PC가 단독으로 y와 얼마나 관련 있는지**”를 보는 **진단/해석용**입니다.
- 하지만 “PCA의 논리(분산 큰 축부터)”에 맞춰 **차원을 점진적으로 늘리며** 예측이 어떻게 바뀌는지 보려면,
  이 노트북처럼 **PC1부터 누적**하는 흐름이 가장 표준적입니다(=PCR).

## 주의(중요)
- PCA는 **X의 분산**을 최대화하는 축이고, **y 예측**을 최대화하는 축이 아닙니다.
  그래서 PC1부터 누적이 “항상” 최적 예측을 보장하지는 않습니다.
- 그럼에도 PCR learning curve는
  - “분산을 얼마나 설명하면 예측이 좋아지는가?”
  - “어느 지점부터 과적합/잡음이 늘어나는가?”
  를 보기 위한 가장 합리적인 실험입니다.

## 출력 구조
`RUN_DIR/` 아래에:
- `종합/` : PCA 축/EVR/재구성 체크 + 모든 target의 누적 PCR 결과 종합
- `target_01_<target명>/` ~ `target_50_<target명>/` : target별 누적 PCR 결과(표 + 그래프)

## 저장 지표
- RMSE, R²
- train_accuracy/test_accuracy: **Pearson correlation** (회귀에서 “정확도” 대신 방향성 포함 지표로 저장)


In [1]:

import os
import re
import datetime as dt
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score


In [2]:

# =========================================================
# 0) 설정 (필요한 부분만 수정)
# =========================================================

# ---- 데이터 경로 ----
DATA_XLSX_PATH     = r"/home/a202192020/맥주데이터실험/data/Supplemental Files and Figure source files.xlsx"
FALLBACK_XLSX_PATH = r"home/a202192020/맥주데이터실험/data/Supplemental Files and Figure source files.xlsx"
CHEM_SHEET = "Supplementary File S1"
SENS_SHEET = "Supplementary File S4"

# ---- 출력 루트 ----
OUT_ROOT = r"/home/a202192020/맥주데이터실험/pca_add/0225/output"

# (A) 자동 timestamp 폴더를 쓰려면 아래 유지
RUN_TAG = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR = os.path.join(OUT_ROOT, RUN_TAG)

# (B) 특정 폴더로 고정하고 싶으면 위 RUN_DIR을 아래처럼 직접 지정:
# RUN_DIR = r"/home/a202192020/맥주데이터실험/pca/0223/pca_y1/output/20260223_174342"

SUMMARY_DIR = os.path.join(RUN_DIR, "종합")
os.makedirs(SUMMARY_DIR, exist_ok=True)

# ---- split ----
TEST_SIZE = 0.30
RANDOM_STATE = 0
STRATIFY_COL = "tasting_category_fine"

# ---- PCA ----
STANDARDIZE_BEFORE_PCA = True
EIG_TOL = 1e-12

# ---- 누적 PCR ----
# 계산량이 부담되면 MAX_K를 줄일 수 있음 (예: 80)
MAX_K = None   # None이면 k_keep까지 전부
PLOT_EVERY_TARGET = True

# ---- CV(선택) ----
# 누적 성능곡선에서 "test"는 최종 검증용이므로,
# k 선택은 가능하면 train 내부 CV로 하길 권장.
DO_CV = True
N_FOLDS = 5

# ---- 저장 ----
PCA_FULL_NPZ   = os.path.join(SUMMARY_DIR, "pca_axes_full_231.npz")
PCA_KEPT_NPZ   = os.path.join(SUMMARY_DIR, "pca_axes_kept_nonzero.npz")
EV_CSV_PATH    = os.path.join(SUMMARY_DIR, "explained_variance_ratio_all_pcs.csv")
RECON_CSV_PATH = os.path.join(SUMMARY_DIR, "reconstruction_check_train_test.csv")

ALLRES_CSV_PATH = os.path.join(SUMMARY_DIR, "cumulative_pcr_all_targets.csv")
BESTK_CSV_PATH  = os.path.join(SUMMARY_DIR, "best_k_per_target.csv")

print("RUN_DIR:", RUN_DIR)
print("SUMMARY_DIR:", SUMMARY_DIR)
print("DO_CV:", DO_CV, "N_FOLDS:", N_FOLDS)


RUN_DIR: /home/a202192020/맥주데이터실험/pca_add/0225/output/20260226_014121
SUMMARY_DIR: /home/a202192020/맥주데이터실험/pca_add/0225/output/20260226_014121/종합
DO_CV: True N_FOLDS: 5


In [3]:

# =========================================================
# 1) Excel 로드 + merge
# =========================================================
xlsx_path = DATA_XLSX_PATH
if not os.path.exists(xlsx_path):
    if os.path.exists(FALLBACK_XLSX_PATH):
        xlsx_path = FALLBACK_XLSX_PATH
        print("⚠️ using fallback:", xlsx_path)
    else:
        raise FileNotFoundError(f"Excel not found:\n- {DATA_XLSX_PATH}\n- {FALLBACK_XLSX_PATH}")

chem_df = pd.read_excel(xlsx_path, sheet_name=CHEM_SHEET)
sens_df = pd.read_excel(xlsx_path, sheet_name=SENS_SHEET)

META_COLS = ["beer", "beer_id", "tasting_category_fine"]
df = chem_df.merge(sens_df, on=META_COLS, how="inner", validate="one_to_one")

feature_cols = [c for c in chem_df.columns if c not in META_COLS]
target_cols  = [c for c in sens_df.columns if c not in META_COLS]

print("merged df:", df.shape)
print("#features:", len(feature_cols), " #targets:", len(target_cols))
assert len(feature_cols) == 231
assert len(target_cols) == 50


merged df: (250, 284)
#features: 231  #targets: 50


In [4]:

# =========================================================
# 2) Train/Test split (175/75) + style stratify
# =========================================================
train_df, test_df = train_test_split(
    df,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    shuffle=True,
    stratify=df[STRATIFY_COL]
)

X_train = train_df[feature_cols].copy()
X_test  = test_df[feature_cols].copy()

Y_train = train_df[target_cols].copy()
Y_test  = test_df[target_cols].copy()

style_train = train_df[STRATIFY_COL].copy()  # CV stratify용

print("train:", X_train.shape, Y_train.shape, "test:", X_test.shape, Y_test.shape)

# 결측치: train 평균으로 대체
impute_means = X_train.mean(axis=0)
X_train = X_train.fillna(impute_means)
X_test  = X_test.fillna(impute_means)


train: (175, 231) (175, 50) test: (75, 231) (75, 50)


In [5]:

# =========================================================
# 3) PCA(공분산 고유분해) — train으로만
#    - 231개 직교축 생성
#    - eig<=tol 축 제거 → kept 축 저장
#    - EVR 저장
# =========================================================

Xtr = X_train.values.astype(float)
Xte = X_test.values.astype(float)

# (A) train 기준 센터링/표준화
mu = Xtr.mean(axis=0)
Xtr_c = Xtr - mu
Xte_c = Xte - mu

if STANDARDIZE_BEFORE_PCA:
    sigma = Xtr_c.std(axis=0, ddof=0)
    sigma_safe = sigma.copy()
    sigma_safe[sigma_safe == 0] = 1.0
    Xtr_cs = Xtr_c / sigma_safe
    Xte_cs = Xte_c / sigma_safe
else:
    sigma_safe = np.ones(Xtr_c.shape[1], dtype=float)
    Xtr_cs = Xtr_c
    Xte_cs = Xte_c

n_train, p = Xtr_cs.shape

# (B) 공분산행렬
S = (Xtr_cs.T @ Xtr_cs) / (n_train - 1)

# (C) 고유분해
eigvals, eigvecs = np.linalg.eigh(S)  # asc
idx = np.argsort(eigvals)[::-1]       # desc
eigvals = np.clip(eigvals[idx], 0, None)
eigvecs = eigvecs[:, idx]

total_var = eigvals.sum()
evr = eigvals / total_var if total_var > 0 else np.zeros_like(eigvals)

keep_mask = eigvals > EIG_TOL
k_keep = int(keep_mask.sum())
kept_full_indices = np.where(keep_mask)[0]  # 0-based in full

V_keep = eigvecs[:, keep_mask]
lam_keep = eigvals[keep_mask]
evr_keep = evr[keep_mask]

print("n_train:", n_train, "p:", p)
print(f"kept components: {k_keep}/{p} (drop {p-k_keep})")

# (D) score
Z_train = Xtr_cs @ V_keep
Z_test  = Xte_cs @ V_keep

# (E) 저장
np.savez(
    PCA_FULL_NPZ,
    feature_cols=np.array(feature_cols, dtype=object),
    mu=mu,
    sigma=sigma_safe,
    standardize_before_pca=np.array([STANDARDIZE_BEFORE_PCA]),
    eig_tol=np.array([EIG_TOL]),
    eigvals=eigvals,
    eigvecs=eigvecs,
    evr=evr,
)
np.savez(
    PCA_KEPT_NPZ,
    feature_cols=np.array(feature_cols, dtype=object),
    mu=mu,
    sigma=sigma_safe,
    standardize_before_pca=np.array([STANDARDIZE_BEFORE_PCA]),
    eig_tol=np.array([EIG_TOL]),
    keep_mask=keep_mask,
    kept_full_indices=kept_full_indices,
    V_keep=V_keep,
    lam_keep=lam_keep,
    evr_keep=evr_keep,
)

# EVR table
ev_df = pd.DataFrame({
    "pc_index_1based_full": np.arange(1, p+1),
    "eigenvalue": eigvals,
    "explained_variance_ratio": evr,
    "explained_variance_ratio_percent": evr*100,
    "kept_nonzero": keep_mask
})
ev_df.to_csv(EV_CSV_PATH, index=False)

print("✅ saved:", PCA_FULL_NPZ)
print("✅ saved:", PCA_KEPT_NPZ)
print("✅ saved:", EV_CSV_PATH)


n_train: 175 p: 231
kept components: 174/231 (drop 57)
✅ saved: /home/a202192020/맥주데이터실험/pca_add/0225/output/20260226_014121/종합/pca_axes_full_231.npz
✅ saved: /home/a202192020/맥주데이터실험/pca_add/0225/output/20260226_014121/종합/pca_axes_kept_nonzero.npz
✅ saved: /home/a202192020/맥주데이터실험/pca_add/0225/output/20260226_014121/종합/explained_variance_ratio_all_pcs.csv


In [6]:

# =========================================================
# 4) 복원력(재구성) 체크: kept 축 투영 → 원공간 복원(train/test)
# =========================================================
def recon_metrics(X_true: np.ndarray, X_hat: np.ndarray) -> dict:
    err = X_true - X_hat
    return {
        "rmse": float(np.sqrt(np.mean(err**2))),
        "mae": float(np.mean(np.abs(err))),
        "max_abs_err": float(np.max(np.abs(err))),
    }

Xtr_hat_cs = Z_train @ V_keep.T
Xte_hat_cs = Z_test  @ V_keep.T
Xtr_hat = Xtr_hat_cs * sigma_safe + mu
Xte_hat = Xte_hat_cs * sigma_safe + mu

m_tr = recon_metrics(Xtr, Xtr_hat)
m_te = recon_metrics(Xte, Xte_hat)

recon_df = pd.DataFrame([{
    "n_train": n_train,
    "n_test": Xte.shape[0],
    "p": p,
    "k_keep": k_keep,
    "standardize_before_pca": STANDARDIZE_BEFORE_PCA,
    "eig_tol": EIG_TOL,
    "train_recon_rmse": m_tr["rmse"],
    "train_recon_mae": m_tr["mae"],
    "train_recon_max_abs_err": m_tr["max_abs_err"],
    "test_recon_rmse": m_te["rmse"],
    "test_recon_mae": m_te["mae"],
    "test_recon_max_abs_err": m_te["max_abs_err"],
}])
recon_df.to_csv(RECON_CSV_PATH, index=False)
print("✅ saved:", RECON_CSV_PATH)

print("Train recon RMSE:", m_tr["rmse"])
print("Test  recon RMSE:", m_te["rmse"])


✅ saved: /home/a202192020/맥주데이터실험/pca_add/0225/output/20260226_014121/종합/reconstruction_check_train_test.csv
Train recon RMSE: 1.6404119125200864e-13
Test  recon RMSE: 544.6602640737248


In [7]:

# =========================================================
# 5) helper: metrics + 폴더명 안전화
# =========================================================
def safe_name(s: str) -> str:
    s = str(s)
    s = re.sub(r"[^0-9A-Za-z가-힣._-]+", "_", s).strip("_")
    return s[:80] if len(s) > 80 else s

def rmse(y_true, y_pred) -> float:
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def pearson_corr(a: np.ndarray, b: np.ndarray) -> float:
    a = np.asarray(a).ravel()
    b = np.asarray(b).ravel()
    if a.size == 0 or a.size != b.size:
        return float("nan")
    if np.std(a) == 0 or np.std(b) == 0:
        return float("nan")
    return float(np.corrcoef(a, b)[0,1])


In [8]:

# =========================================================
# 6) 누적 PCR 실험: PC1 → PC1+PC2 → ... (각 target에 대해)
#    - (선택) train 내부 StratifiedKFold로 CV 성능도 같이 저장
# =========================================================

# k 범위 결정
if MAX_K is None:
    max_k_use = k_keep
else:
    max_k_use = int(min(MAX_K, k_keep))
print("max_k_use:", max_k_use)

# CV splits (style stratified)
if DO_CV:
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    cv_splits = list(skf.split(Z_train[:, :max_k_use], style_train))
else:
    cv_splits = None

all_rows = []
bestk_rows = []

for t_i, target in enumerate(target_cols, start=1):
    target_safe = safe_name(target)
    target_dir = os.path.join(RUN_DIR, f"target_{t_i:02d}_{target_safe}")
    os.makedirs(target_dir, exist_ok=True)

    ytr = Y_train[target].values.astype(float)
    yte = Y_test[target].values.astype(float)

    rows = []
    for k in range(1, max_k_use+1):
        Xk_tr = Z_train[:, :k]
        Xk_te = Z_test[:,  :k]

        model = LinearRegression()
        model.fit(Xk_tr, ytr)

        pred_tr = model.predict(Xk_tr)
        pred_te = model.predict(Xk_te)

        train_rmse = rmse(ytr, pred_tr)
        test_rmse  = rmse(yte, pred_te)
        train_r2 = float(r2_score(ytr, pred_tr))
        test_r2  = float(r2_score(yte, pred_te))
        train_acc = pearson_corr(ytr, pred_tr)
        test_acc  = pearson_corr(yte, pred_te)

        # 누적 EVR (%): 첫 k개 PC가 X 분산을 얼마나 설명하는지
        cum_evr_percent = float(np.sum(evr_keep[:k]) * 100.0)

        out = {
            "target": target,
            "target_index_1based": t_i,
            "k": k,
            "cum_evr_percent": cum_evr_percent,

            "train_rmse": train_rmse,
            "train_r2": train_r2,
            "train_accuracy": train_acc,

            "test_rmse": test_rmse,
            "test_r2": test_r2,
            "test_accuracy": test_acc,
        }

        # ---- CV (선택) ----
        if DO_CV:
            fold_rmses = []
            fold_r2s = []
            for tr_idx, va_idx in cv_splits:
                X_tr = Xk_tr[tr_idx]
                y_tr = ytr[tr_idx]
                X_va = Xk_tr[va_idx]
                y_va = ytr[va_idx]

                m = LinearRegression()
                m.fit(X_tr, y_tr)
                p_va = m.predict(X_va)

                fold_rmses.append(rmse(y_va, p_va))
                fold_r2s.append(float(r2_score(y_va, p_va)))

            out["cv_rmse_mean"] = float(np.mean(fold_rmses))
            out["cv_rmse_std"]  = float(np.std(fold_rmses, ddof=1)) if len(fold_rmses) > 1 else 0.0
            out["cv_r2_mean"]   = float(np.mean(fold_r2s))
            out["cv_r2_std"]    = float(np.std(fold_r2s, ddof=1)) if len(fold_r2s) > 1 else 0.0

        rows.append(out)

    res_df = pd.DataFrame(rows)

    # best k 선정(권장: CV RMSE 최소)
    if DO_CV:
        best_row = res_df.sort_values(["cv_rmse_mean", "k"], ascending=[True, True]).iloc[0]
        best_k = int(best_row["k"])
    else:
        # (권장X) test로 고르면 편향이 생김. 그래도 “분석용”으로는 참고 가능.
        best_row = res_df.sort_values(["test_rmse", "k"], ascending=[True, True]).iloc[0]
        best_k = int(best_row["k"])

    # 저장
    out_csv = os.path.join(target_dir, f"cumulative_pcr_{target_safe}.csv")
    res_df.to_csv(out_csv, index=False)

    # best 요약 텍스트
    out_txt = os.path.join(target_dir, f"best_k_summary_{target_safe}.txt")
    with open(out_txt, "w", encoding="utf-8") as f:
        f.write(f"Target: {target} (index {t_i})\n")
        f.write(f"Best k (selection={'CV_RMSE' if DO_CV else 'TEST_RMSE'}): {best_k}\n")
        f.write(f"Cum EVR(%): {float(best_row['cum_evr_percent']):.6f}\n")
        if DO_CV:
            f.write(f"CV   : RMSE={float(best_row['cv_rmse_mean']):.6f} (std {float(best_row['cv_rmse_std']):.6f}), "
                    f"R2={float(best_row['cv_r2_mean']):.6f}\n")
        f.write(f"Train: RMSE={float(best_row['train_rmse']):.6f}, R2={float(best_row['train_r2']):.6f}, ACC(corr)={float(best_row['train_accuracy']):.6f}\n")
        f.write(f"Test : RMSE={float(best_row['test_rmse']):.6f}, R2={float(best_row['test_r2']):.6f}, ACC(corr)={float(best_row['test_accuracy']):.6f}\n")

    # plot
    if PLOT_EVERY_TARGET:
        plt.figure(figsize=(8,4))
        plt.plot(res_df["k"], res_df["train_r2"], marker="o", linewidth=1, label="train_r2")
        plt.plot(res_df["k"], res_df["test_r2"],  marker="o", linewidth=1, label="test_r2")
        if DO_CV:
            plt.plot(res_df["k"], res_df["cv_r2_mean"], marker="o", linewidth=1, label="cv_r2_mean")
        plt.axvline(best_k, linestyle="--")
        plt.title(f"Cumulative PCR R2 vs k — {target}")
        plt.xlabel("k (number of PCs used: PC1..PCk)")
        plt.ylabel("R2")
        plt.grid(True)
        plt.tight_layout()
        plt.savefig(os.path.join(target_dir, f"cumulative_r2_curve_{target_safe}.png"), dpi=160)
        plt.close()

        plt.figure(figsize=(8,4))
        plt.plot(res_df["k"], res_df["train_rmse"], marker="o", linewidth=1, label="train_rmse")
        plt.plot(res_df["k"], res_df["test_rmse"],  marker="o", linewidth=1, label="test_rmse")
        if DO_CV:
            plt.plot(res_df["k"], res_df["cv_rmse_mean"], marker="o", linewidth=1, label="cv_rmse_mean")
        plt.axvline(best_k, linestyle="--")
        plt.title(f"Cumulative PCR RMSE vs k — {target}")
        plt.xlabel("k (number of PCs used: PC1..PCk)")
        plt.ylabel("RMSE")
        plt.grid(True)
        plt.tight_layout()
        plt.savefig(os.path.join(target_dir, f"cumulative_rmse_curve_{target_safe}.png"), dpi=160)
        plt.close()

    # 종합 누적
    all_rows.append(res_df)
    bestk_rows.append({
        "target": target,
        "target_index_1based": t_i,
        "best_k": best_k,
        "best_cum_evr_percent": float(best_row["cum_evr_percent"]),
        "best_train_rmse": float(best_row["train_rmse"]),
        "best_train_r2": float(best_row["train_r2"]),
        "best_test_rmse": float(best_row["test_rmse"]),
        "best_test_r2": float(best_row["test_r2"]),
        "best_train_accuracy": float(best_row["train_accuracy"]),
        "best_test_accuracy": float(best_row["test_accuracy"]),
        **({"best_cv_rmse_mean": float(best_row["cv_rmse_mean"]),
            "best_cv_r2_mean": float(best_row["cv_r2_mean"])} if DO_CV else {})
    })

    if t_i % 5 == 0:
        print(f"processed targets: {t_i}/{len(target_cols)}")

# ---- 종합 저장 ----
all_df = pd.concat(all_rows, axis=0, ignore_index=True)
bestk_df = pd.DataFrame(bestk_rows)

all_df.to_csv(ALLRES_CSV_PATH, index=False)
bestk_df.to_csv(BESTK_CSV_PATH, index=False)

print("✅ saved:", ALLRES_CSV_PATH)
print("✅ saved:", BESTK_CSV_PATH)

display(bestk_df.sort_values("best_test_r2", ascending=False).head(15))


max_k_use: 174


/home/a202192020/ys1_env/lib/python3.8/site-packages/sklearn/model_selection/_split.py:737: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(


processed targets: 5/50
processed targets: 10/50
processed targets: 15/50
processed targets: 20/50
processed targets: 25/50
processed targets: 30/50
processed targets: 35/50
processed targets: 40/50
processed targets: 45/50
processed targets: 50/50
✅ saved: /home/a202192020/맥주데이터실험/pca_add/0225/output/20260226_014121/종합/cumulative_pcr_all_targets.csv
✅ saved: /home/a202192020/맥주데이터실험/pca_add/0225/output/20260226_014121/종합/best_k_per_target.csv


,target,target_index_1based,best_k,best_cum_evr_percent,best_train_rmse,best_train_r2,best_test_rmse,best_test_r2,best_train_accuracy,best_test_accuracy,best_cv_rmse_mean,best_cv_r2_mean
31,bitternes,32,12,53.816778,0.425953,0.641868,0.507635,0.564397,0.801167,0.752299,0.472292,0.513396
0,A_malt_all,1,24,67.993120,0.455692,0.539163,0.638855,0.373095,0.734277,0.621386,0.561881,0.264377
46,aftertaste,47,11,52.222987,0.408495,0.540674,0.535281,0.333902,0.735305,0.604595,0.435826,0.455981
32,sweetness,33,14,56.811583,0.569590,0.319456,0.535050,0.326100,0.565204,0.586219,0.632634,0.115238
8,A_hops_noble,9,4,33.206098,0.439988,0.304567,0.489237,0.186652,0.551876,0.475894,0.443581,0.280823
9,A_hops_woody,10,4,33.206098,0.321236,0.141769,0.417923,-0.027067,0.376522,0.154721,0.330096,0.048335
4,A_malt_burn,5,1,14.323116,0.517063,0.010156,0.950532,-0.029167,0.100779,0.256729,0.501292,-0.069590
12,A_esters_isoaa,13,1,14.323116,0.534770,0.007613,0.550704,-0.046791,0.087251,-0.203826,0.533587,-0.020657
41,clove,42,1,14.323116,0.600052,0.024309,0.431807,-0.049199,0.155914,-0.007847,0.517049,0.019693
44,barnyard,45,15,58.145723,0.390936,0.214894,0.478023,-0.064476,0.463566,0.143739,0.419483,0.064916
